In [1]:
from langgraph.graph import StateGraph, START, MessagesState
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

In [15]:

load_dotenv()
model=ChatGoogleGenerativeAI(model="gemini-3.8-flash")

In [17]:
def call_model(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

In [18]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

In [19]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [20]:
config = {"configurable": {"thread_id": "thread-1"}}
config2 = {"configurable": {"thread_id": "thread-2"}}

In [21]:

graph.invoke({"messages": [{"role": "user", "content": "Hi! My name is Chanchal."}]},config)

{'messages': [HumanMessage(content='Hi! My name is Chanchal.', additional_kwargs={}, response_metadata={}, id='ff7be845-3e6f-4b2b-b3c3-906208e622e5'),
  AIMessage(content=[{'type': 'text', 'text': "Hello Chanchal! It's nice to meet you. How can I help you today?", 'extras': {'signature': 'EvIFCu8FAWkUfROu3YuXOM5ZMeoooeK9klhsM2DonMozI5tUHBPLTtUt76bo2qprbms5sNVjFDi0H6DVdlBD7csMGedXfZdq+OkRWU5mTvNiqxh/y15namrhO7zECdDHFIcZgmEzM0MemwVT8RcaHidGsgzxdJ/K514sldo1smqWYcIlOtjikBBJKO9a13I7rYTH7S6tp5UVhcsVfoDfVYWKCXDCWCzdwLCDZMsVJkURb8y0Dt6ORYx82m7tWBOYplac1yVrCu+CRr//iePjqohaN7XBB8k1gquvXp6Pz0sV1swxJhfTDatYLVtdm9AC2MyEVpu8ugUTFkkRry5nQU6Bin1zfDqwz/EUBE753wWc27tgPve9KVuBVyyU3qVVp3Z1IuOVWzcmyYuDGF3JV0On6r95AJrH8gDJVoUKAUQzWg2zVzIjgN3SnmjyHNbcmdZ0NH3eCuxD4uPmg6NJF8N2tjpCUpOsbGCVgNY49uUB+osCvmUy0rClYcGqXmngXg6Ju9sO1GE2xnvI7cUCx+29HyV1HEg8O9CG9ChNm8gPnCq3HMVOWkragd43Hc+TMoHzegOQtmGJVMtB1EaQrrsfxpQZIlgRlPGASNWNwGNp6EuvaYeyc+dIS1eeJ2EDSokYr2Q0OfBthPDSvuGzdG1ikJFkGZ+1QF7/0oCjE8has3InCvAWTKsaDk/fIQ32YAvAwZ

In [23]:
graph.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config2)

{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='2c2755cc-9dcb-4349-a147-83176f84950f'),
  AIMessage(content=[{'type': 'text', 'text': "I don't know your name because you haven't told me yet! What should I call you?", 'extras': {'signature': 'EqsFCqgFAWkUfRM9+tfg5uUfZeBco0MZJqjszC7epZlqFnSqdqlqnviSDSDdjlBF07e+70O7nBq/ntzRWcdMBGHBAQ2iprfDRDd188bA6dP+Zmq0lQRWJB1suvfipeeTm9j/No+2sLXY4evuGriq7NZf4xX7zEuLd8ABDA7B5ZKk2sEb7dGYgN1JjJHukROIPbNgOZN/D/qX0UE2CHT6fCFC0HQ4C9jHdG41GA4Hi7ZnD53AKYd3E92A2dBZyhL5AL5Zsrry7GxWwtl1CvqhxaBIq7+bRjxkgO8qkz6xwx20t0IgJAeXwvJKXC9ir9HzOe2l9KmiBEd60S3g2/Uso83d29aOI05mxDormgica7NiusrfYJjNVfpyhm6nBmOYbUtQWV/KeUhmqbjbFrka3aPNSJUTF67VxBFtfpSy/McoEeJcLTKaHTGZG1vJmWjYnZFiwYHWWlPC1GGetC+pYMKZkQ8Qaa5x2EqbZewzRhR2GKe6w1TPmFyePeGKr7jPB/Z3Ih5Jb9PXkvvrLvfsZa9An0TMlTeQXE8QaAEJN38Yh1MYDwpnU7H0uZj4p1G4VUul/qGSGLdwIHwiRp7VmcXOj5vRMvD8HcbC/a00UCOwHRCKt41mDAsWtHXNb46DBxEiUyqvTxstaedwrz9bTYFAGBwAJBJ9YvnEg8QGH4ex1Gfahi+zXnnaErW8bow

In [25]:

snap = graph.get_state(config2)
vals = snap.values
for m in vals.get("messages", []):
        print("-", type(m).__name__, ":", m.content)

- HumanMessage : What is my name?
- AIMessage : [{'type': 'text', 'text': "I don't know your name because you haven't told me yet! What should I call you?", 'extras': {'signature': 'EqsFCqgFAWkUfRM9+tfg5uUfZeBco0MZJqjszC7epZlqFnSqdqlqnviSDSDdjlBF07e+70O7nBq/ntzRWcdMBGHBAQ2iprfDRDd188bA6dP+Zmq0lQRWJB1suvfipeeTm9j/No+2sLXY4evuGriq7NZf4xX7zEuLd8ABDA7B5ZKk2sEb7dGYgN1JjJHukROIPbNgOZN/D/qX0UE2CHT6fCFC0HQ4C9jHdG41GA4Hi7ZnD53AKYd3E92A2dBZyhL5AL5Zsrry7GxWwtl1CvqhxaBIq7+bRjxkgO8qkz6xwx20t0IgJAeXwvJKXC9ir9HzOe2l9KmiBEd60S3g2/Uso83d29aOI05mxDormgica7NiusrfYJjNVfpyhm6nBmOYbUtQWV/KeUhmqbjbFrka3aPNSJUTF67VxBFtfpSy/McoEeJcLTKaHTGZG1vJmWjYnZFiwYHWWlPC1GGetC+pYMKZkQ8Qaa5x2EqbZewzRhR2GKe6w1TPmFyePeGKr7jPB/Z3Ih5Jb9PXkvvrLvfsZa9An0TMlTeQXE8QaAEJN38Yh1MYDwpnU7H0uZj4p1G4VUul/qGSGLdwIHwiRp7VmcXOj5vRMvD8HcbC/a00UCOwHRCKt41mDAsWtHXNb46DBxEiUyqvTxstaedwrz9bTYFAGBwAJBJ9YvnEg8QGH4ex1Gfahi+zXnnaErW8bowu37PMGATbjF/Ef5kIRXYd+VP7H4Lb9XUhRSMvru+1oohXh5OaFSHcvib8kO43Epe6ZJwNQCRw9/d1qGyGliFJ4E9EHUdN54rFknVCqrbPTf7gE0Wf0n